# Segmentation Diagnostic v1

**Purpose:** Threshold calibration for the 4-class label scheme before full patch regeneration.

**Workflow:**
1. Load image, specify one good Z per timepoint manually
2. Extract ~5 patches per timepoint from those planes via droplet centroids
3. Test `npc_ring_threshold_k` and `nucleus_threshold_k` interactively
4. Visualize raw vs. thresholded in grid comparisons

**Image:** `(T=10, Z=20, C=3, Y=3889, X=5732)` — channels: 0=Membrane, 1=NLS, 2=NPC

**Classes:** 0=background, 1=droplet, 2=NPC ring, 3=nucleus interior

## Section 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import tifffile
from skimage import filters, morphology, measure, segmentation
from skimage.morphology import disk, binary_closing, remove_small_objects, remove_small_holes
from scipy import ndimage as ndi

print('Imports OK')

## Section 2 — Config

In [ ]:
@dataclass
class DiagConfig:
    # ── Image ─────────────────────────────────────────────────────────────
    image_path: str = "/home/tdeibert/Data/Machine_Learning_Dev/Images/control_extract_1.1.tif"

    # Channel indices (C axis = axis 2)
    membrane_channel_idx: int = 0
    nls_channel_idx:      int = 1
    npc_channel_idx:      int = 2

    pixel_size_um: float = 0.108

    # ── Manual Z-plane selection ───────────────────────────────────────────
    # Specify one good in-focus Z per timepoint.
    # Key = timepoint index (0-based), Value = Z index (0-based).
    # Edit these after inspecting the image.
    focus_z_per_t: dict = field(default_factory=lambda: {
        0:  10,   # t=0  — adjust as needed
        1:  11,   # t=1
        2:  12,   # t=2
        3:  13,   # t=3
        4:  14,   # t=4
        5:  14,   # t=5
        6:  15,   # t=6
        7:  15,   # t=7  — known good from prior diagnostics
        8:  15,   # t=8
        9:  16,   # t=9
    })

    # ── Patch extraction ──────────────────────────────────────────────────
    patch_size:         int   = 256    # pixels, square
    patches_per_t:      int   = 5      # number of patches to extract per timepoint
    patch_jitter_px:    int   = 20     # random centroid jitter (same as v6.1.7)
    seed:               int   = 42

    # ── Droplet segmentation (matches v6.1.7) ─────────────────────────────
    droplet_blur_sigma:         float = 4.0
    droplet_local_block:        int   = 151   # must be odd
    droplet_local_offset:       float = -0.02
    droplet_min_area_um2:       float = 3000.0
    droplet_max_area_um2:       float = 80000.0
    droplet_closing_radius_um:  float = 2.0
    droplet_hole_area_um2:      float = 5000.0

    # ── NPC ring detection ────────────────────────────────────────────────
    # Threshold = mean + k * std within each droplet's ring zone.
    # Start at 0.5 and sweep to 2.0 — see Section 5.
    npc_ring_threshold_k:   float = 0.5
    droplet_focus_erosion_px: int = 20   # ring zone width: droplet edge inward

    # ── Nucleus segmentation ──────────────────────────────────────────────
    # Threshold = mean + k * std within each droplet interior.
    # Start at 1.0 and sweep — see Section 6.
    nucleus_threshold_k:        float = 1.0
    nucleus_blur_sigma:         float = 2.0
    min_nucleus_area_um2:       float = 50.0
    max_nucleus_area_um2:       float = 3000.0
    nucleus_closing_radius_um:  float = 1.5
    nucleus_hole_area_um2:      float = 200.0


cfg = DiagConfig()

print('Config:')
print(f'  image_path            : {cfg.image_path}')
print(f'  patch_size            : {cfg.patch_size}')
print(f'  patches_per_t         : {cfg.patches_per_t}')
print(f'  npc_ring_threshold_k  : {cfg.npc_ring_threshold_k}')
print(f'  nucleus_threshold_k   : {cfg.nucleus_threshold_k}')
print(f'  droplet_focus_erosion : {cfg.droplet_focus_erosion_px} px')
print(f'  focus_z_per_t         : {cfg.focus_z_per_t}')

## Section 3 — Load Image

In [ ]:
img = tifffile.memmap(cfg.image_path)
print(f'Shape : {img.shape}')   # expect (10, 20, 3, 3889, 5732)
print(f'Dtype : {img.dtype}')

n_t, n_z, n_c, height, width = img.shape

## Section 4 — Segmentation Functions (matches v6.1.7)

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────

def normalize_01(arr):
    """Normalize a 2D array to [0, 1] float32."""
    arr = arr.astype(np.float32)
    mn, mx = arr.min(), arr.max()
    if mx == mn:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)


def um_to_px(um, pixel_size_um):
    return max(1, int(round(um / pixel_size_um)))


# ── Droplet segmentation ──────────────────────────────────────────────────

def segment_droplet_from_npc(npc2d, pixel_size_um=None, cfg=cfg):
    """
    Segment droplets from the NPC channel using adaptive local thresholding.
    Matches the v6.1.7 implementation exactly.
    """
    if pixel_size_um is None:
        pixel_size_um = cfg.pixel_size_um

    npc_f = normalize_01(np.asarray(npc2d, dtype=np.float32))
    blurred = filters.gaussian(npc_f, sigma=cfg.droplet_blur_sigma, preserve_range=True)

    local_thresh = filters.threshold_local(
        blurred,
        block_size=cfg.droplet_local_block,
        offset=cfg.droplet_local_offset
    )
    binary = blurred > local_thresh

    # Morphological cleanup
    closing_px = um_to_px(cfg.droplet_closing_radius_um, pixel_size_um)
    binary = binary_closing(binary, disk(closing_px))

    hole_px2 = cfg.droplet_hole_area_um2 / (pixel_size_um ** 2)
    binary = remove_small_holes(binary, area_threshold=int(hole_px2))

    min_px2 = cfg.droplet_min_area_um2 / (pixel_size_um ** 2)
    max_px2 = cfg.droplet_max_area_um2 / (pixel_size_um ** 2)
    labeled = measure.label(binary)
    for prop in measure.regionprops(labeled):
        if prop.area < min_px2 or prop.area > max_px2:
            binary[labeled == prop.label] = False

    return binary.astype(bool)


# ── NPC ring detection ────────────────────────────────────────────────────

def detect_npc_rings_per_droplet(npc2d, droplet_mask, pixel_size_um=None,
                                  erosion_px=None, threshold_k=None, cfg=cfg):
    """
    Detect NPC rings per droplet by thresholding the NPC channel within
    the annular zone between the droplet edge and its eroded interior.

    Threshold = mean + k * std within each droplet's ring zone.
    Robust to varying signal intensity across timepoints.
    Matches the v6.1.7 implementation exactly.
    """
    if pixel_size_um is None:
        pixel_size_um = cfg.pixel_size_um
    if erosion_px is None:
        erosion_px = cfg.droplet_focus_erosion_px
    if threshold_k is None:
        threshold_k = cfg.npc_ring_threshold_k

    npc_f = normalize_01(np.asarray(npc2d, dtype=np.float32))
    labeled_droplets = measure.label(droplet_mask)
    npc_ring_mask = np.zeros_like(droplet_mask, dtype=bool)

    # Distance transform to define ring zone
    dist = ndi.distance_transform_edt(droplet_mask.astype(bool))
    interior_mask = dist >= erosion_px
    ring_mask_global = droplet_mask.astype(bool) & ~interior_mask

    # Per-droplet adaptive thresholding within ring zone
    ring_labels = labeled_droplets * ring_mask_global.astype(np.int32)
    ids = np.unique(ring_labels)[1:]  # skip background

    if len(ids) == 0:
        return npc_ring_mask

    means = ndi.mean(npc_f, ring_labels, ids)
    stds  = ndi.standard_deviation(npc_f, ring_labels, ids)

    for did, mu, sd in zip(ids, means, stds):
        thr = mu + threshold_k * sd
        ring_pixels = (ring_labels == did)
        npc_ring_mask[ring_pixels & (npc_f >= thr)] = True

    return npc_ring_mask


# ── Nucleus segmentation ──────────────────────────────────────────────────

def segment_nucleus_per_droplet(nls2d, droplet_mask, pixel_size_um=None,
                                 threshold_k=None, cfg=cfg):
    """
    Per-droplet nucleus segmentation using NLS channel.
    Threshold = mean + k * std within each droplet interior.
    Matches the v6.1.7 vectorised implementation.
    """
    if pixel_size_um is None:
        pixel_size_um = cfg.pixel_size_um
    if threshold_k is None:
        threshold_k = cfg.nucleus_threshold_k

    nls_f = normalize_01(np.asarray(nls2d, dtype=np.float32))
    blurred = filters.gaussian(nls_f, sigma=cfg.nucleus_blur_sigma, preserve_range=True)

    labeled_droplets = measure.label(droplet_mask)
    nucleus_mask = np.zeros_like(droplet_mask, dtype=bool)

    ids = np.unique(labeled_droplets)[1:]
    if len(ids) == 0:
        return nucleus_mask

    # Vectorised per-droplet thresholding
    means = ndi.mean(blurred, labeled_droplets, ids)
    stds  = ndi.standard_deviation(blurred, labeled_droplets, ids)

    for did, mu, sd in zip(ids, means, stds):
        thr = mu + threshold_k * sd
        nucleus_mask[labeled_droplets == did] |= (
            (blurred >= thr) & (labeled_droplets == did)
        )

    # Morphological cleanup
    closing_px = um_to_px(cfg.nucleus_closing_radius_um, pixel_size_um)
    nucleus_mask = binary_closing(nucleus_mask, disk(closing_px))

    hole_px2 = cfg.nucleus_hole_area_um2 / (pixel_size_um ** 2)
    nucleus_mask = remove_small_holes(nucleus_mask, area_threshold=int(hole_px2))

    min_px2 = cfg.min_nucleus_area_um2 / (pixel_size_um ** 2)
    max_px2 = cfg.max_nucleus_area_um2 / (pixel_size_um ** 2)
    labeled_nuc = measure.label(nucleus_mask)
    for prop in measure.regionprops(labeled_nuc):
        if prop.area < min_px2 or prop.area > max_px2:
            nucleus_mask[labeled_nuc == prop.label] = False

    # Enforce nucleus inside droplet
    nucleus_mask = nucleus_mask & droplet_mask.astype(bool)

    return nucleus_mask.astype(bool)


# ── 4-class label assembly ────────────────────────────────────────────────

def make_4class_label(npc2d, nls2d, droplet_mask, pixel_size_um=None,
                       npc_k=None, nuc_k=None, cfg=cfg):
    """
    Assemble a 4-class integer label map from three binary masks.

    Class priority (later classes overwrite earlier):
        0 = background
        1 = droplet interior
        2 = NPC ring
        3 = nucleus interior

    Returns
    -------
    label       : 2D uint8 array with values 0-3
    ring_mask   : 2D bool, NPC ring pixels
    nuc_mask    : 2D bool, nucleus interior pixels
    """
    if pixel_size_um is None:
        pixel_size_um = cfg.pixel_size_um
    if npc_k is None:
        npc_k = cfg.npc_ring_threshold_k
    if nuc_k is None:
        nuc_k = cfg.nucleus_threshold_k

    label = np.zeros(droplet_mask.shape, dtype=np.uint8)
    label[droplet_mask.astype(bool)] = 1

    ring_mask = detect_npc_rings_per_droplet(
        npc2d, droplet_mask, pixel_size_um=pixel_size_um,
        threshold_k=npc_k, cfg=cfg)
    label[ring_mask] = 2

    nuc_mask = segment_nucleus_per_droplet(
        nls2d, droplet_mask, pixel_size_um=pixel_size_um,
        threshold_k=nuc_k, cfg=cfg)
    label[nuc_mask] = 3

    return label, ring_mask, nuc_mask


# ── Patch extraction ──────────────────────────────────────────────────────

def jitter_center(cy, cx, jitter_px, height, width, patch_size, rng):
    half = patch_size // 2
    cy = int(cy) + rng.integers(-jitter_px, jitter_px + 1)
    cx = int(cx) + rng.integers(-jitter_px, jitter_px + 1)
    cy = np.clip(cy, half, height - half)
    cx = np.clip(cx, half, width  - half)
    return cy, cx


def extract_patch_2d(arr2d, cy, cx, patch_size):
    half = patch_size // 2
    return arr2d[cy - half : cy + half, cx - half : cx + half]


def extract_plane_patch(img, t, z, cy, cx, patch_size):
    """Extract all channels for a given (t, z) plane as a (C, H, W) patch."""
    half = patch_size // 2
    return np.asarray(img[t, z, :, cy - half : cy + half, cx - half : cx + half])


print('Functions defined OK')

## Section 5 — Extract Test Patches

For each timepoint: segment droplets from the manually selected Z plane, 
pick `patches_per_t` droplet centroids, extract patches.

In [ ]:
rng = np.random.default_rng(cfg.seed)

# Stored patches: list of dicts, one per patch
# Keys: t, z, cy, cx, img_patch (C,H,W), droplet_mask (H,W)
patches = []

for t in range(n_t):
    z = cfg.focus_z_per_t[t]

    npc2d = np.asarray(img[t, z, cfg.npc_channel_idx], dtype=np.float32)

    droplet_mask = segment_droplet_from_npc(npc2d, cfg=cfg)
    labeled      = measure.label(droplet_mask)
    props        = measure.regionprops(labeled)

    if len(props) == 0:
        print(f't={t} z={z}: no droplets detected — check focus_z_per_t')
        continue

    # Sort by area descending, take up to patches_per_t largest droplets
    props_sorted = sorted(props, key=lambda p: p.area, reverse=True)
    selected     = props_sorted[:cfg.patches_per_t]

    for prop in selected:
        cy_raw, cx_raw = prop.centroid
        cy, cx = jitter_center(
            cy_raw, cx_raw, cfg.patch_jitter_px,
            height, width, cfg.patch_size, rng
        )

        img_patch = extract_plane_patch(img, t, z, cy, cx, cfg.patch_size)
        if img_patch.shape != (n_c, cfg.patch_size, cfg.patch_size):
            continue  # edge case: centroid too close to border

        dm_patch  = extract_patch_2d(droplet_mask, cy, cx, cfg.patch_size)

        patches.append(dict(t=t, z=z, cy=cy, cx=cx,
                            img_patch=img_patch, droplet_mask=dm_patch))

    print(f't={t:2d}  z={z:2d}  droplets found: {len(props):3d}  '
          f'patches taken: {len(selected)}')

print(f'\nTotal patches: {len(patches)}')

## Section 6 — Diagnostic: NPC Ring Threshold Sweep

For each patch, show: NPC raw | ring mask at k=0.5 | k=1.0 | k=1.5 | k=2.0

Run once per timepoint — set `DIAG_T` to the timepoint you want to inspect.

**What to look for:** At the correct `k`, the ring mask should be a clean 
thin annulus hugging the inside of the droplet boundary. Too low → noise 
fills the interior. Too high → ring disappears.

In [ ]:
DIAG_T      = 7        # timepoint to inspect
NPC_K_SWEEP = [0.5, 1.0, 1.5, 2.0]  # k values to test

t_patches = [p for p in patches if p['t'] == DIAG_T]
if not t_patches:
    print(f'No patches for t={DIAG_T}')
else:
    n_cols = 1 + len(NPC_K_SWEEP)  # raw NPC + one column per k
    n_rows = len(t_patches)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 3.5, n_rows * 3.5),
        squeeze=False
    )
    fig.suptitle(f'NPC Ring Threshold Sweep — t={DIAG_T}  '
                 f'z={cfg.focus_z_per_t[DIAG_T]}  '
                 f'erosion={cfg.droplet_focus_erosion_px}px',
                 fontsize=13, fontweight='bold')

    col_labels = ['NPC raw'] + [f'ring k={k}' for k in NPC_K_SWEEP]
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=10)

    for row, p in enumerate(t_patches):
        npc_patch = p['img_patch'][cfg.npc_channel_idx]
        dm_patch  = p['droplet_mask']

        # Column 0: NPC raw with droplet boundary overlay
        ax = axes[row][0]
        ax.imshow(normalize_01(npc_patch), cmap='gray', interpolation='nearest')
        boundary = segmentation.find_boundaries(dm_patch, mode='outer')
        ax.contour(boundary, levels=[0.5], colors='cyan', linewidths=0.8)
        ax.set_ylabel(f'patch {row}\ncy={p["cy"]} cx={p["cx"]}', fontsize=7)
        ax.axis('off')

        # Columns 1-N: ring mask for each k
        for col, k in enumerate(NPC_K_SWEEP, start=1):
            ring = detect_npc_rings_per_droplet(
                npc_patch, dm_patch, threshold_k=k, cfg=cfg)

            ax = axes[row][col]
            # Composite: NPC gray + ring overlay in green
            rgb = np.stack([normalize_01(npc_patch)] * 3, axis=-1)
            rgb[ring, 0] = 0.0
            rgb[ring, 1] = 1.0
            rgb[ring, 2] = 0.0
            ax.imshow(rgb, interpolation='nearest')
            ax.contour(boundary, levels=[0.5], colors='cyan', linewidths=0.8)
            ring_frac = ring.sum() / max(dm_patch.sum(), 1)
            ax.set_xlabel(f'{ring.sum()} px  ({ring_frac:.2%})', fontsize=7)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'/tmp/npc_ring_sweep_t{DIAG_T}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: /tmp/npc_ring_sweep_t{DIAG_T}.png')

## Section 7 — Diagnostic: Nucleus Threshold Sweep

For each patch, show: NLS raw | nucleus mask at k=0.5 | k=1.0 | k=1.5 | k=2.0

**What to look for:** At the correct `k`, the nucleus mask should be a 
single clean blob inside the droplet interior, with no bleed into the ring 
zone or over-segmentation into multiple fragments.

In [ ]:
NUC_K_SWEEP = [0.5, 1.0, 1.5, 2.0]

t_patches = [p for p in patches if p['t'] == DIAG_T]
if not t_patches:
    print(f'No patches for t={DIAG_T}')
else:
    n_cols = 1 + len(NUC_K_SWEEP)
    n_rows = len(t_patches)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 3.5, n_rows * 3.5),
        squeeze=False
    )
    fig.suptitle(f'Nucleus Threshold Sweep — t={DIAG_T}  '
                 f'z={cfg.focus_z_per_t[DIAG_T]}',
                 fontsize=13, fontweight='bold')

    col_labels = ['NLS raw'] + [f'nuc k={k}' for k in NUC_K_SWEEP]
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=10)

    for row, p in enumerate(t_patches):
        nls_patch = p['img_patch'][cfg.nls_channel_idx]
        dm_patch  = p['droplet_mask']

        ax = axes[row][0]
        ax.imshow(normalize_01(nls_patch), cmap='gray', interpolation='nearest')
        boundary = segmentation.find_boundaries(dm_patch, mode='outer')
        ax.contour(boundary, levels=[0.5], colors='cyan', linewidths=0.8)
        ax.set_ylabel(f'patch {row}', fontsize=7)
        ax.axis('off')

        for col, k in enumerate(NUC_K_SWEEP, start=1):
            nuc = segment_nucleus_per_droplet(
                nls_patch, dm_patch, threshold_k=k, cfg=cfg)

            ax = axes[row][col]
            rgb = np.stack([normalize_01(nls_patch)] * 3, axis=-1)
            rgb[nuc, 0] = 1.0
            rgb[nuc, 1] = 0.3
            rgb[nuc, 2] = 0.0
            ax.imshow(rgb, interpolation='nearest')
            ax.contour(boundary, levels=[0.5], colors='cyan', linewidths=0.8)
            nuc_frac = nuc.sum() / max(dm_patch.sum(), 1)
            ax.set_xlabel(f'{nuc.sum()} px  ({nuc_frac:.2%})', fontsize=7)
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'/tmp/nuc_sweep_t{DIAG_T}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: /tmp/nuc_sweep_t{DIAG_T}.png')

## Section 8 — Diagnostic: Full 4-Class Label Grid

After choosing candidate `k` values from Sections 6 & 7, assemble the 
full 4-class label and compare raw channels side-by-side.

**Grid per patch:** Membrane | NLS | NPC | 4-class label overlay

**Label colormap:** black=bg, blue=droplet, green=NPC ring, red=nucleus

In [ ]:
# ── Set final candidate k values here after inspecting Sections 6 & 7 ────
NPC_K_FINAL = 1.0   # adjust based on Section 6 output
NUC_K_FINAL = 1.0   # adjust based on Section 7 output

# 4-class colormap: 0=black, 1=dodger blue, 2=lime, 3=crimson
cmap_4class = ListedColormap(['#000000', '#1E90FF', '#00FF00', '#DC143C'])

t_patches = [p for p in patches if p['t'] == DIAG_T]
if not t_patches:
    print(f'No patches for t={DIAG_T}')
else:
    n_cols = 5  # membrane | nls | npc | label | label+npc overlay
    n_rows = len(t_patches)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 3.5, n_rows * 3.5),
        squeeze=False
    )
    fig.suptitle(
        f'4-Class Label Assembly — t={DIAG_T}  z={cfg.focus_z_per_t[DIAG_T]}  '
        f'npc_k={NPC_K_FINAL}  nuc_k={NUC_K_FINAL}',
        fontsize=13, fontweight='bold'
    )

    col_labels = ['Membrane (ch0)', 'NLS (ch1)', 'NPC (ch2)',
                  '4-class label', 'NPC + label overlay']
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=10)

    for row, p in enumerate(t_patches):
        mem_patch = p['img_patch'][cfg.membrane_channel_idx]
        nls_patch = p['img_patch'][cfg.nls_channel_idx]
        npc_patch = p['img_patch'][cfg.npc_channel_idx]
        dm_patch  = p['droplet_mask']

        label, ring_mask, nuc_mask = make_4class_label(
            npc_patch, nls_patch, dm_patch,
            npc_k=NPC_K_FINAL, nuc_k=NUC_K_FINAL, cfg=cfg
        )

        boundary = segmentation.find_boundaries(dm_patch, mode='outer')

        def _show(ax, arr, title=None, cmap='gray'):
            ax.imshow(arr if arr.ndim == 3 else normalize_01(arr),
                      cmap=cmap, interpolation='nearest',
                      vmin=0, vmax=1 if cmap == 'gray' else None)
            ax.contour(boundary, levels=[0.5], colors='cyan', linewidths=0.6)
            ax.axis('off')

        _show(axes[row][0], mem_patch)
        _show(axes[row][1], nls_patch)
        _show(axes[row][2], npc_patch)

        # 4-class label
        axes[row][3].imshow(label, cmap=cmap_4class, vmin=0, vmax=3,
                            interpolation='nearest')
        axes[row][3].axis('off')

        # NPC gray + label class overlay (semi-transparent)
        ax = axes[row][4]
        ax.imshow(normalize_01(npc_patch), cmap='gray', interpolation='nearest')
        overlay = np.zeros((*npc_patch.shape, 4), dtype=np.float32)
        # droplet: blue 30%
        overlay[dm_patch & ~ring_mask & ~nuc_mask] = [0.1, 0.4, 1.0, 0.25]
        # NPC ring: green 70%
        overlay[ring_mask]  = [0.0, 1.0, 0.0, 0.65]
        # nucleus: red 60%
        overlay[nuc_mask]   = [1.0, 0.1, 0.1, 0.60]
        ax.imshow(overlay, interpolation='nearest')
        ax.axis('off')

        # Row label: class pixel counts
        counts = {c: int((label == c).sum()) for c in range(4)}
        axes[row][0].set_ylabel(
            f'patch {row}\nbg:{counts[0]} dr:{counts[1]}\n'
            f'ring:{counts[2]} nuc:{counts[3]}',
            fontsize=7
        )

    # Legend
    legend_patches = [
        mpatches.Patch(color='#000000', label='0 background'),
        mpatches.Patch(color='#1E90FF', label='1 droplet'),
        mpatches.Patch(color='#00FF00', label='2 NPC ring'),
        mpatches.Patch(color='#DC143C', label='3 nucleus'),
    ]
    fig.legend(handles=legend_patches, loc='lower center',
               ncol=4, fontsize=9, frameon=True,
               bbox_to_anchor=(0.5, -0.01))

    plt.tight_layout()
    plt.savefig(f'/tmp/4class_label_t{DIAG_T}_npcK{NPC_K_FINAL}_nucK{NUC_K_FINAL}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved to /tmp/')

## Section 9 — Cross-Timepoint Summary Grid

Once you have settled on `NPC_K_FINAL` and `NUC_K_FINAL`, run this cell.

Shows one representative patch per timepoint with the 4-class overlay so 
you can verify the thresholds hold across the full time series 
(early = no nucleus, late = full ring + nucleus).

In [ ]:
# Uses NPC_K_FINAL and NUC_K_FINAL from Section 8

# One patch per timepoint (first patch = largest droplet)
rep_patches = []
for t in range(n_t):
    tp = [p for p in patches if p['t'] == t]
    if tp:
        rep_patches.append(tp[0])

n_rows = len(rep_patches)
n_cols = 4  # NPC raw | NLS raw | 4-class label | overlay

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(n_cols * 3.5, n_rows * 3.2),
    squeeze=False
)
fig.suptitle(
    f'Cross-Timepoint Summary  npc_k={NPC_K_FINAL}  nuc_k={NUC_K_FINAL}\n'
    f'One representative patch per timepoint (largest droplet)',
    fontsize=12, fontweight='bold'
)

for ax, lbl in zip(axes[0], ['NPC (ch2)', 'NLS (ch1)',
                               '4-class label', 'NPC + overlay']):
    ax.set_title(lbl, fontsize=10)

for row, p in enumerate(rep_patches):
    t = p['t']
    nls_patch = p['img_patch'][cfg.nls_channel_idx]
    npc_patch = p['img_patch'][cfg.npc_channel_idx]
    dm_patch  = p['droplet_mask']

    label, ring_mask, nuc_mask = make_4class_label(
        npc_patch, nls_patch, dm_patch,
        npc_k=NPC_K_FINAL, nuc_k=NUC_K_FINAL, cfg=cfg
    )

    boundary = segmentation.find_boundaries(dm_patch, mode='outer')

    axes[row][0].imshow(normalize_01(npc_patch), cmap='gray', interpolation='nearest')
    axes[row][0].contour(boundary, levels=[0.5], colors='cyan', linewidths=0.6)
    axes[row][0].set_ylabel(f't={t}  z={p["z"]}', fontsize=9)

    axes[row][1].imshow(normalize_01(nls_patch), cmap='gray', interpolation='nearest')
    axes[row][1].contour(boundary, levels=[0.5], colors='cyan', linewidths=0.6)

    axes[row][2].imshow(label, cmap=cmap_4class, vmin=0, vmax=3,
                        interpolation='nearest')

    ax = axes[row][3]
    ax.imshow(normalize_01(npc_patch), cmap='gray', interpolation='nearest')
    overlay = np.zeros((*npc_patch.shape, 4), dtype=np.float32)
    overlay[dm_patch & ~ring_mask & ~nuc_mask] = [0.1, 0.4, 1.0, 0.25]
    overlay[ring_mask]  = [0.0, 1.0, 0.0, 0.65]
    overlay[nuc_mask]   = [1.0, 0.1, 0.1, 0.60]
    ax.imshow(overlay, interpolation='nearest')

    for ax in axes[row]:
        ax.axis('off')

    counts = {c: int((label == c).sum()) for c in range(4)}
    axes[row][2].set_title(
        f'ring:{counts[2]} nuc:{counts[3]}', fontsize=7
    )

legend_patches = [
    mpatches.Patch(color='#000000', label='0 background'),
    mpatches.Patch(color='#1E90FF', label='1 droplet'),
    mpatches.Patch(color='#00FF00', label='2 NPC ring'),
    mpatches.Patch(color='#DC143C', label='3 nucleus'),
]
fig.legend(handles=legend_patches, loc='lower center',
           ncol=4, fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(f'/tmp/cross_timepoint_summary_npcK{NPC_K_FINAL}_nucK{NUC_K_FINAL}.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('Saved to /tmp/')

## Section 10 — Erosion Width Diagnostic (Optional)

If the ring mask looks too wide or too narrow, sweep `droplet_focus_erosion_px`.

Use the `NPC_K_FINAL` value already established in Section 6.

In [ ]:
EROSION_SWEEP = [10, 15, 20, 25, 30]  # pixels
EROSION_DIAG_PATCH_IDX = 0            # which patch from DIAG_T to use

t_patches = [p for p in patches if p['t'] == DIAG_T]
if not t_patches:
    print(f'No patches for t={DIAG_T}')
else:
    p = t_patches[EROSION_DIAG_PATCH_IDX]
    npc_patch = p['img_patch'][cfg.npc_channel_idx]
    dm_patch  = p['droplet_mask']
    boundary  = segmentation.find_boundaries(dm_patch, mode='outer')

    n_cols = 1 + len(EROSION_SWEEP)
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 3.5, 4), squeeze=False)
    fig.suptitle(
        f'Ring Width (erosion) Sweep — t={DIAG_T}  npc_k={NPC_K_FINAL}',
        fontsize=12, fontweight='bold'
    )

    axes[0][0].imshow(normalize_01(npc_patch), cmap='gray', interpolation='nearest')
    axes[0][0].contour(boundary, levels=[0.5], colors='cyan', linewidths=0.8)
    axes[0][0].set_title('NPC raw')
    axes[0][0].axis('off')

    for col, epx in enumerate(EROSION_SWEEP, start=1):
        ring = detect_npc_rings_per_droplet(
            npc_patch, dm_patch,
            erosion_px=epx,
            threshold_k=NPC_K_FINAL,
            cfg=cfg
        )
        ax = axes[0][col]
        rgb = np.stack([normalize_01(npc_patch)] * 3, axis=-1)
        rgb[ring, 0] = 0.0
        rgb[ring, 1] = 1.0
        rgb[ring, 2] = 0.0
        ax.imshow(rgb, interpolation='nearest')
        ax.contour(boundary, levels=[0.5], colors='cyan', linewidths=0.8)
        ring_um = epx * cfg.pixel_size_um
        ax.set_title(f'{epx}px ({ring_um:.1f}µm)\n{ring.sum()} ring px', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(f'/tmp/erosion_sweep_t{DIAG_T}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved to /tmp/')